# Nhánh NER thứ ba — ViHealthBERT trên VietMed-NER

**Notebook release: `2026-09-22-v1-vihealthbert-local-gpu`**

Notebook này sao chép quy trình của `giai_doan_13_ner_finetune_phobert/NER_PhoBERT_RunAll.ipynb` (v9) và chỉ đổi base model sang `demdecuong/vihealthbert-base-syllable`. Đây là BERT tiếng Việt được pretrain thêm trên văn bản y tế, ở mức âm tiết, khớp với dữ liệu VietMed-NER.

Các phần giữ nguyên để so sánh công bằng với PhoBERT:
- Cùng dataset `leduckhai/VietMed-NER`, cùng cách chia train/validation/test.
- Cùng lưới 18 cấu hình (learning rate × epoch × weight decay), sau đó chạy 3 seed `42/123/2024` cho cấu hình tốt nhất.
- Cùng cách căn nhãn theo subtoken đầu, cùng evaluator `seqeval`, và test chỉ được đọc ở cell đánh giá.
- Cùng 500 transcript ASR (tái sử dụng file kết quả của PhoBERT), nên retention so sánh được trực tiếp.

Các phần khác so với bản Colab:
- Chạy trên **GPU local**. Notebook dừng ngay nếu không thấy CUDA.
- Dùng mixed precision bf16 và TF32 để train nhanh hơn.
- Checkpoint trung gian của 18 trial sweep được xoá để tiết kiệm ổ đĩa. Checkpoint của 3 seed được giữ lại.

Cách chạy: xem `README.md` cùng thư mục. Đặt `SMOKE_TEST = True` ở cell cấu hình để chạy thử vài phút trước khi chạy thật.

In [1]:
# 0. Kiểm tra GPU — notebook này bắt buộc chạy trên CUDA
import platform
import torch

print('Python:', platform.python_version())
print('PyTorch:', torch.__version__, '| CUDA build:', torch.version.cuda)
if not torch.cuda.is_available():
    raise RuntimeError(
        'Không thấy CUDA GPU. Kiểm tra: (1) kernel Jupyter đang dùng .venv của thư mục này, '
        '(2) đã cài torch bản cu128 (xem setup_env.sh), (3) chạy `nvidia-smi` trong WSL.'
    )

gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
arch_list = torch.cuda.get_arch_list()
total_vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'GPU: {gpu_name} | compute capability {major}.{minor} | VRAM {total_vram_gb:.1f} GB')
print('Kiến trúc PyTorch hỗ trợ:', arch_list)
if f'sm_{major}{minor}' not in arch_list and f'compute_{major}{minor}' not in arch_list:
    raise RuntimeError(
        f'Bản PyTorch này không có kernel cho sm_{major}{minor}. '
        'RTX 50xx (Blackwell) cần torch>=2.7 bản cu128.'
    )

# TF32 tăng tốc matmul trên Ampere trở lên, sai số không đáng kể với fine-tuning.
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
_probe = torch.randn(1024, 1024, device='cuda')
print('GPU matmul smoke test:', float((_probe @ _probe).abs().mean()) > 0)
del _probe

Python: 3.12.14
PyTorch: 2.8.0+cu128 | CUDA build: 12.8
GPU: NVIDIA GeForce RTX 5060 Ti | compute capability 12.0 | VRAM 15.9 GB
Kiến trúc PyTorch hỗ trợ: ['sm_70', 'sm_75', 'sm_80', 'sm_86', 'sm_90', 'sm_100', 'sm_120']
GPU matmul smoke test: True


In [2]:
# 1. Cấu hình
import json
import os
from pathlib import Path
from typing import Any

import numpy as np
from datasets import load_dataset
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorForTokenClassification,
    Trainer,
    TrainingArguments,
)

# Đặt True để chạy thử nhanh (1 trial × 1 epoch trên tập con). Nhớ đặt lại False để chạy thật.
SMOKE_TEST = False

BASE_MODEL = 'demdecuong/vihealthbert-base-syllable'
MODEL_KEY = 'vihealthbert'
DATASET_ID = 'leduckhai/VietMed-NER'
DEVICE = 'cuda'
SEED = 42
SEEDS = (42, 123, 2024)
MAX_LENGTH = 256  # ViHealthBERT có max_position_embeddings=258 → tối đa 256 token + 2 special

# bf16 trên GPU hỗ trợ (Ampere trở lên). Đặt False nếu muốn fp32 hoàn toàn như bản Colab T4.
USE_BF16 = torch.cuda.is_bf16_supported()
# Sweep chỉ cần metric validation; xoá weight của 18 trial để tiết kiệm ~10 GB ổ đĩa.
KEEP_SWEEP_WEIGHTS = False

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / 'NER_ViHealthBERT_RunAll.ipynb').exists():
    raise RuntimeError(f'Hãy mở Jupyter từ thư mục chứa notebook. Hiện tại: {NOTEBOOK_DIR}')
suffix = '-smoke' if SMOKE_TEST else ''
OUTPUT_DIR = NOTEBOOK_DIR / 'outputs' / f'vihealthbert-vietmed-ner{suffix}'
RESULT_DIR = NOTEBOOK_DIR / f'ketqua{suffix}'
PHOBERT_RESULT_DIR = NOTEBOOK_DIR.parent / 'giai_doan_13_ner_finetune_phobert' / 'ketqua'
# Tuỳ chọn: thư mục đã giải nén phobert-best-seed.zip (seed_123). Nếu có, PhoBERT được chấm lại
# trên máy này. Nếu không, notebook đọc số liệu PhoBERT từ PHOBERT_RESULT_DIR.
PHOBERT_CHECKPOINT = os.environ.get('PHOBERT_CHECKPOINT', '')
XLMR_REPO, XLMR_SUBFOLDER = 'leduckhai/VietMed-NER', 'xlm-roberta-base-VietMed-NER'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)


def to_jsonable(value):
    '''Convert NumPy/Torch scalar and nested report values to JSON types.'''
    if isinstance(value, dict):
        return {str(key): to_jsonable(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_jsonable(item) for item in value]
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().tolist()
    return value

print('SMOKE_TEST:', SMOKE_TEST)
print('Base model:', BASE_MODEL)
print('Mixed precision bf16:', USE_BF16)
print('Output (checkpoint):', OUTPUT_DIR)
print('Kết quả (JSON/bảng):', RESULT_DIR)
print('PhoBERT checkpoint:', PHOBERT_CHECKPOINT or '(không có, dùng số liệu đã lưu)')

/home/minhchanh/UIT/Capstone/VoDoCo/giai_doan_15_ner_finetune_vihealthbert/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


SMOKE_TEST: False
Base model: demdecuong/vihealthbert-base-syllable
Mixed precision bf16: True
Output (checkpoint): /home/minhchanh/UIT/Capstone/VoDoCo/giai_doan_15_ner_finetune_vihealthbert/outputs/vihealthbert-vietmed-ner
Kết quả (JSON/bảng): /home/minhchanh/UIT/Capstone/VoDoCo/giai_doan_15_ner_finetune_vihealthbert/ketqua
PhoBERT checkpoint: (không có, dùng số liệu đã lưu)


## 2. Tải dữ liệu và chuẩn bị nhãn

Code giống hệt notebook PhoBERT. Chỉ khác một điểm: bỏ cột `audio` vì NER chỉ cần text, giúp không phải giải mã audio. Kết quả không thay đổi.

In [3]:
dataset = load_dataset(DATASET_ID)
if 'audio' in dataset['train'].column_names:
    dataset = dataset.remove_columns('audio')
label_column = 'labels' if 'labels' in dataset['train'].column_names else 'tags'
feature = dataset['train'].features[label_column].feature

# Label vocabulary is derived from train only; test is not touched during setup or tuning.
if hasattr(feature, 'names'):
    label_names = list(feature.names)
else:
    label_names = sorted({
        str(label)
        for example in dataset['train']
        for label in example[label_column]
    })

label2id = {label: index for index, label in enumerate(label_names)}
id2label = {index: label for label, index in label2id.items()}

def label_to_id(label: Any) -> int:
    if isinstance(label, int) and hasattr(feature, 'names'):
        label = feature.int2str(label)
    label = str(label)
    if label not in label2id:
        raise ValueError(f'Unknown label {label!r}; expected {label_names}')
    return label2id[label]

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
# Hub repo không có tokenizer_config.json nên model_max_length mặc định rất lớn.
tokenizer.model_max_length = MAX_LENGTH
print('Tokenizer:', tokenizer.__class__.__name__)
print('Tokenizer fast:', getattr(tokenizer, 'is_fast', False))

def encode_words(words, current_tokenizer, row_labels=None):
    raw_ids = []
    raw_word_ids = []
    raw_labels = [] if row_labels is not None else None
    for word_index, word in enumerate(words):
        subtoken_ids = current_tokenizer.encode(word, add_special_tokens=False)
        if not subtoken_ids:
            continue
        raw_ids.extend(subtoken_ids)
        raw_word_ids.extend([word_index] * len(subtoken_ids))
        if raw_labels is not None:
            raw_labels.extend([label_to_id(row_labels[word_index])] + [-100] * (len(subtoken_ids) - 1))

    input_ids = current_tokenizer.build_inputs_with_special_tokens(raw_ids)
    raw_start = None
    for start in range(len(input_ids) - len(raw_ids) + 1):
        if input_ids[start:start + len(raw_ids)] == raw_ids:
            raw_start = start
            break
    if raw_start is None:
        raise ValueError('Could not locate raw tokens in special-token sequence')

    input_ids = input_ids[:MAX_LENGTH]
    raw_end = min(raw_start + len(raw_ids), len(input_ids))
    aligned_word_ids = [None] * len(input_ids)
    aligned_labels = [-100] * len(input_ids) if raw_labels is not None else None
    for raw_index, position in enumerate(range(raw_start, raw_end)):
        aligned_word_ids[position] = raw_word_ids[raw_index]
        if aligned_labels is not None:
            aligned_labels[position] = raw_labels[raw_index]

    return {
        'input_ids': input_ids,
        'attention_mask': [1] * len(input_ids),
    }, aligned_word_ids, aligned_labels

def tokenize_and_align(examples):
    batch = {'input_ids': [], 'attention_mask': [], 'labels': []}
    for words, row_labels in zip(examples['words'], examples[label_column]):
        encoded, _, aligned_labels = encode_words(words, tokenizer, row_labels)
        for key in encoded:
            batch[key].append(encoded[key])
        batch['labels'].append(aligned_labels)
    return batch

tokenized = {
    split_name: dataset[split_name].map(
        tokenize_and_align,
        batched=True,
        remove_columns=dataset[split_name].column_names,
    )
    for split_name in ('train', 'validation')
}
test_examples = dataset['test']
if SMOKE_TEST:
    tokenized['train'] = tokenized['train'].select(range(256))
    tokenized['validation'] = tokenized['validation'].select(range(128))
    test_examples = test_examples.select(range(100))

smoke_example = tokenized['train'][0]
assert len(smoke_example['input_ids']) == len(smoke_example['attention_mask']) == len(smoke_example['labels'])
assert any(label != -100 for label in smoke_example['labels']), 'No aligned gold labels found'
print('Tokenization smoke test: passed')
print('Tuning splits:', {name: len(split) for name, split in tokenized.items()})
print('Test split is reserved for the final evaluation cells.')
print('Label column:', label_column)
print('Labels:', label_names)

Tokenizer: PhobertTokenizer
Tokenizer fast: False
Tokenization smoke test: passed
Tuning splits: {'train': 4616, 'validation': 1154}
Test split is reserved for the final evaluation cells.
Label column: labels
Labels: ['0', 'B-AGE', 'B-DATETIME', 'B-DIAGNOSTICS', 'B-DISEASESYMTOM', 'B-DRUGCHEMICAL', 'B-FOODDRINK', 'B-GENDER', 'B-LOCATION', 'B-MEDDEVICETECHNIQUE', 'B-OCCUPATION', 'B-ORGAN', 'B-ORGANIZATION', 'B-PERSONALCARE', 'B-PREVENTIVEMED', 'B-SURGERY', 'B-TRANSPORTATION', 'B-TREATMENT', 'B-UNITCALIBRATOR', 'I-AGE', 'I-DATETIME', 'I-DIAGNOSTICS', 'I-DISEASESYMTOM', 'I-DRUGCHEMICAL', 'I-FOODDRINK', 'I-GENDER', 'I-LOCATION', 'I-MEDDEVICETECHNIQUE', 'I-OCCUPATION', 'I-ORGAN', 'I-ORGANIZATION', 'I-PERSONALCARE', 'I-PREVENTIVEMED', 'I-SURGERY', 'I-TRANSPORTATION', 'I-TREATMENT', 'I-UNITCALIBRATOR']


In [4]:
# Thống kê tokenizer trên train: số subtoken/âm tiết và tỉ lệ <unk> (dùng cho phần phân tích trong báo cáo)
def tokenizer_stats(current_tokenizer, examples, limit=2000):
    words = [word for example in examples.select(range(min(limit, len(examples)))) for word in example['words']]
    pieces = [current_tokenizer.encode(word, add_special_tokens=False) for word in words]
    unk_id = current_tokenizer.unk_token_id
    return {
        'tokenizer': current_tokenizer.__class__.__name__,
        'num_words': len(words),
        'subtokens_per_word': sum(len(p) for p in pieces) / len(words),
        'unk_word_rate': sum(1 for p in pieces if unk_id in p) / len(words),
        'truncated_train_rows': sum(1 for row in tokenized['train'] if len(row['input_ids']) >= MAX_LENGTH),
    }

vihealthbert_tokenizer_stats = tokenizer_stats(tokenizer, dataset['train'])
print(json.dumps(vihealthbert_tokenizer_stats, indent=2))

{
  "tokenizer": "PhobertTokenizer",
  "num_words": 48490,
  "subtokens_per_word": 1.0147040626933388,
  "unk_word_rate": 0.0,
  "truncated_train_rows": 0
}


## 3. Fine-tune ViHealthBERT: sweep 18 cấu hình và 3 seed

Mỗi trial ghi một dòng vào `validation_trials.jsonl`. Nếu kernel bị ngắt, chạy lại cell này thì các trial đã xong sẽ được bỏ qua (resume). Mỗi trial còn ghi thời gian train và VRAM đỉnh.

In [ ]:
import gc
import inspect
import itertools
import shutil
import time
from statistics import mean, pstdev

from seqeval.metrics import f1_score, precision_score, recall_score


def compute_metrics(eval_prediction):
    predictions, labels = eval_prediction
    predicted_ids = np.argmax(predictions, axis=2)
    true_predictions, true_labels = [], []
    for prediction, label_row in zip(predicted_ids, labels):
        pred_tags, gold_tags = [], []
        for predicted_id, gold_id in zip(prediction, label_row):
            if gold_id == -100:
                continue
            pred_tags.append(id2label[int(predicted_id)])
            gold_tags.append(id2label[int(gold_id)])
        true_predictions.append(pred_tags)
        true_labels.append(gold_tags)
    return {
        'precision': float(precision_score(true_labels, true_predictions, zero_division=0)),
        'recall': float(recall_score(true_labels, true_predictions, zero_division=0)),
        'f1': float(f1_score(true_labels, true_predictions, zero_division=0)),
    }


def build_trainer(model, output_dir, learning_rate, epochs, weight_decay, seed):
    training_kwargs = dict(
        output_dir=str(output_dir), learning_rate=learning_rate,
        per_device_train_batch_size=16, per_device_eval_batch_size=16,
        num_train_epochs=epochs, weight_decay=weight_decay,
        lr_scheduler_type='linear', warmup_ratio=0.1,
        save_strategy='epoch', save_total_limit=1,
        load_best_model_at_end=True, metric_for_best_model='f1',
        greater_is_better=True, logging_steps=50, seed=seed,
        report_to='none',
        # GPU local
        bf16=USE_BF16, tf32=True,
        dataloader_num_workers=2, dataloader_pin_memory=True,
    )
    training_params = inspect.signature(TrainingArguments.__init__).parameters
    if 'eval_strategy' in training_params:
        training_kwargs['eval_strategy'] = 'epoch'
    else:
        training_kwargs['evaluation_strategy'] = 'epoch'
    if 'save_only_model' in training_params:
        training_kwargs['save_only_model'] = True  # không lưu optimizer state vào checkpoint trung gian
    trainer_args = TrainingArguments(**training_kwargs)
    trainer_kwargs = dict(
        model=model, args=trainer_args,
        train_dataset=tokenized['train'], eval_dataset=tokenized['validation'],
        data_collator=DataCollatorForTokenClassification(tokenizer),
        compute_metrics=compute_metrics,
    )
    if 'processing_class' in inspect.signature(Trainer.__init__).parameters:
        trainer_kwargs['processing_class'] = tokenizer
    else:
        trainer_kwargs['tokenizer'] = tokenizer
    return Trainer(**trainer_kwargs)


def train_trial(trial_id, learning_rate, epochs, weight_decay, seed, keep_weights=True):
    trial_dir = OUTPUT_DIR / trial_id
    trial_dir.mkdir(parents=True, exist_ok=True)
    torch.cuda.reset_peak_memory_stats()
    started = time.time()
    model = AutoModelForTokenClassification.from_pretrained(
        BASE_MODEL, num_labels=len(label_names), id2label=id2label, label2id=label2id
    )
    trainer = build_trainer(model, trial_dir, learning_rate, epochs, weight_decay, seed)
    trainer.train()
    metrics = to_jsonable(trainer.evaluate(tokenized['validation']))
    if keep_weights:
        trainer.save_model(str(trial_dir))
        tokenizer.save_pretrained(str(trial_dir))
    for checkpoint_dir in trial_dir.glob('checkpoint-*'):
        shutil.rmtree(checkpoint_dir, ignore_errors=True)
    result = {
        'trial_id': trial_id, 'base_model': BASE_MODEL, 'learning_rate': learning_rate, 'epochs': epochs,
        'weight_decay': weight_decay, 'warmup_ratio': 0.1,
        'scheduler': 'linear', 'seed': seed, 'bf16': USE_BF16, 'validation': metrics,
        'checkpoint': str(trial_dir) if keep_weights else None,
        'seconds': round(time.time() - started, 2),
        'peak_vram_gb': round(torch.cuda.max_memory_allocated() / 1024**3, 2),
        'gpu': gpu_name,
    }
    del trainer, model
    gc.collect()
    torch.cuda.empty_cache()
    return result


if SMOKE_TEST:
    trial_specs = [('smoke_lr3e-05_ep1_wd0.05', 3e-5, 1, 0.05)]
    seeds_to_run = (SEED,)
else:
    learning_rates = [5e-6, 1e-5, 3e-5]
    epoch_options = [4, 6, 8]
    weight_decays = [0.01, 0.05]
    trial_specs = [
        (f'validation_lr{lr:.0e}_ep{epochs}_wd{wd}', lr, epochs, wd)
        for lr, epochs, wd in itertools.product(learning_rates, epoch_options, weight_decays)
    ]
    seeds_to_run = SEEDS

sweep_path = OUTPUT_DIR / 'validation_trials.jsonl'
completed_trials = {}
if sweep_path.exists():
    for line in sweep_path.read_text(encoding='utf-8').splitlines():
        if line.strip():
            item = json.loads(line)
            completed_trials[item['trial_id']] = item
for index, (trial_id, lr, epochs, wd) in enumerate(trial_specs, start=1):
    if trial_id in completed_trials:
        print('Resume: skip', trial_id)
        continue
    print(f'[{index}/{len(trial_specs)}] Training', trial_id)
    result = train_trial(trial_id, lr, epochs, wd, seed=SEED, keep_weights=KEEP_SWEEP_WEIGHTS)
    completed_trials[trial_id] = result
    with sweep_path.open('a', encoding='utf-8') as handle:
        handle.write(json.dumps(to_jsonable(result), ensure_ascii=False) + '\n')
    print(f"Completed {trial_id} validation F1={result['validation']['eval_f1']:.4f} "
          f"({result['seconds']:.0f}s, VRAM {result['peak_vram_gb']} GB)")

trial_results = sorted(completed_trials.values(), key=lambda item: item['validation']['eval_f1'], reverse=True)
assert len(trial_results) == len(trial_specs), f'Expected {len(trial_specs)} trials, got {len(trial_results)}'
best_config = trial_results[0]
(OUTPUT_DIR / 'validation_sweep.json').write_text(
    json.dumps(to_jsonable({'trials': trial_results, 'best_config': best_config}), ensure_ascii=False, indent=2),
    encoding='utf-8',
)
print('Best validation config:', json.dumps(to_jsonable(best_config), ensure_ascii=False, indent=2))

seed_path = OUTPUT_DIR / 'best_config_seeds.jsonl'
seed_results = {}
if seed_path.exists():
    for line in seed_path.read_text(encoding='utf-8').splitlines():
        if line.strip():
            item = json.loads(line)
            seed_results[item['seed']] = item
for seed in seeds_to_run:
    if seed in seed_results:
        print('Resume: skip seed', seed)
        continue
    seed_id = f"seed_{seed}_lr{best_config['learning_rate']:.0e}_ep{best_config['epochs']}_wd{best_config['weight_decay']}"
    print('Training', seed_id)
    result = train_trial(
        seed_id, best_config['learning_rate'], best_config['epochs'],
        best_config['weight_decay'], seed, keep_weights=True,
    )
    seed_results[seed] = result
    with seed_path.open('a', encoding='utf-8') as handle:
        handle.write(json.dumps(to_jsonable(result), ensure_ascii=False) + '\n')
    print(f"Completed seed {seed} validation F1={result['validation']['eval_f1']:.4f}")

seed_results_list = sorted(seed_results.values(), key=lambda item: item['validation']['eval_f1'], reverse=True)
assert len(seed_results_list) == len(seeds_to_run)
seed_f1_values = [item['validation']['eval_f1'] for item in seed_results_list]
best_seed = seed_results_list[0]
seed_summary = {
    'best_config': best_config,
    'seeds': seed_results_list,
    'validation_f1_mean': mean(seed_f1_values),
    'validation_f1_std_population': pstdev(seed_f1_values),
    'selected_seed': best_seed,
}
(OUTPUT_DIR / 'best_config_seeds.json').write_text(
    json.dumps(to_jsonable(seed_summary), ensure_ascii=False, indent=2), encoding='utf-8'
)
for name in ('validation_trials.jsonl', 'validation_sweep.json', 'best_config_seeds.jsonl', 'best_config_seeds.json'):
    shutil.copy2(OUTPUT_DIR / name, RESULT_DIR / name)
print('Seed validation mean/std:', seed_summary['validation_f1_mean'], seed_summary['validation_f1_std_population'])
print('Selected seed checkpoint:', best_seed['checkpoint'])

[1/18] Training validation_lr5e-06_ep4_wd0.01


Some weights of RobertaForTokenClassification were not initialized from the model checkpoint at demdecuong/vihealthbert-base-syllable and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## 4. Chấm trên test (gold): ViHealthBERT và XLM-RoBERTa, cộng PhoBERT nếu có checkpoint

Seed của ViHealthBERT được chọn **chỉ bằng validation**. Cell này chấm cả 3 seed trên test để báo cáo mean ± std, nhưng số chính thức là của seed đã chọn. XLM-R được chấm lại trên máy này để đối chiếu với kết quả cũ (F1 56,78%).

In [ ]:
from seqeval.metrics import classification_report


def load_ner(path_or_repo, subfolder=None):
    kwargs = {'subfolder': subfolder} if subfolder else {}
    model = AutoModelForTokenClassification.from_pretrained(path_or_repo, **kwargs).to(DEVICE)
    model_tokenizer = AutoTokenizer.from_pretrained(path_or_repo, **kwargs)
    model_tokenizer.model_max_length = MAX_LENGTH
    return model, model_tokenizer


def free_gpu():
    gc.collect()
    torch.cuda.empty_cache()


def evaluate_checkpoint(eval_model, eval_tokenizer, examples):
    eval_model.eval()
    model_id2label = {int(key): value for key, value in eval_model.config.id2label.items()}
    predictions, references = [], []
    for example in examples:
        encoded, aligned_word_ids, _ = encode_words(example['words'], eval_tokenizer)
        model_inputs = {
            key: torch.tensor([value], dtype=torch.long, device=DEVICE)
            for key, value in encoded.items()
        }
        with torch.inference_mode():
            logits = eval_model(**model_inputs).logits
        predicted_ids = logits.argmax(dim=-1)[0].detach().cpu().tolist()
        row_predictions, row_references = [], []
        seen_words = set()
        for token_id, word_id in zip(predicted_ids, aligned_word_ids):
            if word_id is None or word_id in seen_words:
                continue
            if word_id >= len(example[label_column]):
                break
            seen_words.add(word_id)
            row_predictions.append(model_id2label[int(token_id)])
            gold = example[label_column][word_id]
            if isinstance(gold, int) and hasattr(feature, 'names'):
                gold = feature.int2str(gold)
            row_references.append(str(gold))
        predictions.append(row_predictions)
        references.append(row_references)
    report = classification_report(references, predictions, output_dict=True, zero_division=0)
    # Dataset dùng nhãn '0' (số không) thay vì 'O', nên seqeval coi '0' là entity loại '_' và tính vào micro.
    # Giữ số gốc để so với kết quả PhoBERT cũ, đồng thời tính thêm bản chuẩn sau khi đổi '0' → 'O'.
    to_o = lambda rows: [['O' if tag == '0' else tag for tag in row] for row in rows]
    references_o, predictions_o = to_o(references), to_o(predictions)
    return to_jsonable({
        'precision': float(precision_score(references, predictions, zero_division=0)),
        'recall': float(recall_score(references, predictions, zero_division=0)),
        'f1': float(f1_score(references, predictions, zero_division=0)),
        'entity_only': {
            'precision': float(precision_score(references_o, predictions_o, zero_division=0)),
            'recall': float(recall_score(references_o, predictions_o, zero_division=0)),
            'f1': float(f1_score(references_o, predictions_o, zero_division=0)),
            'macro_f1': float(f1_score(references_o, predictions_o, average='macro', zero_division=0)),
        },
        'classification_report': report,
        'num_examples': len(examples),
    })


def fmt(values):
    return f"P={values['precision']:.4f} R={values['recall']:.4f} F1={values['f1']:.4f} n={values['num_examples']}"


gold_metrics = {}

xlmr_model, xlmr_tokenizer = load_ner(XLMR_REPO, XLMR_SUBFOLDER)
gold_metrics['baseline_xlm_roberta'] = evaluate_checkpoint(xlmr_model, xlmr_tokenizer, test_examples)
print('baseline_xlm_roberta:', fmt(gold_metrics['baseline_xlm_roberta']))
del xlmr_model, xlmr_tokenizer
free_gpu()

phobert_available = bool(PHOBERT_CHECKPOINT) and Path(PHOBERT_CHECKPOINT, 'config.json').exists()
if phobert_available:
    phobert_model, phobert_tokenizer = load_ner(PHOBERT_CHECKPOINT)
    gold_metrics['phobert_best_validation_seed'] = evaluate_checkpoint(phobert_model, phobert_tokenizer, test_examples)
    print('phobert_best_validation_seed (chấm lại):', fmt(gold_metrics['phobert_best_validation_seed']))
    del phobert_model, phobert_tokenizer
    free_gpu()
else:
    saved = json.loads((PHOBERT_RESULT_DIR / 'ner_gold_comparison.json').read_text(encoding='utf-8'))
    gold_metrics['phobert_best_validation_seed'] = saved['metrics']['phobert_best_validation_seed']
    print('phobert_best_validation_seed (số liệu đã lưu):', fmt(gold_metrics['phobert_best_validation_seed']))

vihealthbert_seed_test = {}
for seed_result in seed_results_list:
    seed_model, seed_tokenizer = load_ner(seed_result['checkpoint'])
    vihealthbert_seed_test[seed_result['seed']] = evaluate_checkpoint(seed_model, seed_tokenizer, test_examples)
    print(f"vihealthbert seed {seed_result['seed']}:", fmt(vihealthbert_seed_test[seed_result['seed']]))
    del seed_model, seed_tokenizer
    free_gpu()
gold_metrics['vihealthbert_best_validation_seed'] = vihealthbert_seed_test[best_seed['seed']]

seed_test_f1 = [values['f1'] for values in vihealthbert_seed_test.values()]
gold_payload = {
    'dataset': DATASET_ID, 'split': 'test', 'base_model': BASE_MODEL,
    'best_config': best_config, 'seed_summary': seed_summary,
    'metrics': gold_metrics,
    'vihealthbert_all_seeds_test': {
        'per_seed': vihealthbert_seed_test,
        'f1_mean': mean(seed_test_f1),
        'f1_std_population': pstdev(seed_test_f1),
    },
    'phobert_source': 'recomputed' if phobert_available else str(PHOBERT_RESULT_DIR / 'ner_gold_comparison.json'),
    'note': 'All models use the same test split, manual first-subtoken alignment and seqeval evaluator. '
            'ViHealthBERT seed was selected using validation only; per-seed test scores are reported for variance only.',
}
(RESULT_DIR / 'ner_gold_comparison.json').write_text(
    json.dumps(to_jsonable(gold_payload), ensure_ascii=False, indent=2), encoding='utf-8'
)
print('\nViHealthBERT test F1 3 seed: mean', round(mean(seed_test_f1), 4), 'std', round(pstdev(seed_test_f1), 4))

## 5. Pipeline ASR → NER trên 500 audio test (entity retention)

Mặc định dùng lại 500 transcript Whisper đã lưu ở `giai_doan_13_ner_finetune_phobert/ketqua/asr_ner_comparison.jsonl`. Như vậy cả 3 model NER nhận **đúng cùng một transcript** với lần chạy PhoBERT, và không cần tải khoảng 1 GB audio. Đặt `RUN_ASR = True` nếu muốn chạy lại Whisper trên GPU.

Retention đo mức nhất quán giữa entity dự đoán trên transcript ASR và entity dự đoán trên transcript chuẩn. **Đây không phải recall so với nhãn gold.**

In [ ]:
from collections import Counter
from transformers import pipeline

RUN_ASR = False
ASR_SAMPLES = 20 if SMOKE_TEST else 500
cached_asr_path = PHOBERT_RESULT_DIR / 'asr_ner_comparison.jsonl'

if not RUN_ASR and cached_asr_path.exists():
    cached_rows = [json.loads(line) for line in cached_asr_path.read_text(encoding='utf-8').splitlines() if line.strip()]
    asr_pairs = [
        {'index': row['index'], 'reference_transcript': row['reference_transcript'],
         'asr_transcript': row['asr_transcript'], 'wer': row['wer']}
        for row in cached_rows[:ASR_SAMPLES]
    ]
    asr_source = f'cached: {cached_asr_path}'
else:
    from datasets import Audio
    from jiwer import wer
    from transformers import AutoFeatureExtractor, AutoModelForSpeechSeq2Seq, AutoProcessor

    ASR_REPO = 'leduckhai/MultiMed-ST'
    ASR_SUBFOLDER = 'asr/whisper-small-vietnamese/checkpoint-5000'
    ASR_PROCESSOR_SUBFOLDER = 'asr/whisper-small-vietnamese'
    asr_model = AutoModelForSpeechSeq2Seq.from_pretrained(
        ASR_REPO, subfolder=ASR_SUBFOLDER, dtype=torch.float16,
        low_cpu_mem_usage=True, use_safetensors=True,
    ).to('cuda')
    asr_processor = AutoProcessor.from_pretrained(ASR_REPO, subfolder=ASR_PROCESSOR_SUBFOLDER)
    asr_tokenizer = getattr(asr_processor, 'tokenizer', asr_processor)
    asr_feature_extractor = getattr(asr_processor, 'feature_extractor', None) or \
        AutoFeatureExtractor.from_pretrained(ASR_REPO, subfolder=ASR_PROCESSOR_SUBFOLDER)
    asr_pipe = pipeline(
        'automatic-speech-recognition', model=asr_model, tokenizer=asr_tokenizer,
        feature_extractor=asr_feature_extractor, dtype=torch.float16, device=0,
    )
    audio_dataset = load_dataset('leduckhai/VietMed', split='test')
    audio_dataset = audio_dataset.cast_column('audio', Audio(sampling_rate=16000, decode=True))
    asr_pairs = []
    for index, row in enumerate(audio_dataset.select(range(min(ASR_SAMPLES, len(audio_dataset))))):
        audio = row['audio']
        transcript = asr_pipe(
            {'raw': audio['array'], 'sampling_rate': audio['sampling_rate']},
            generate_kwargs={'language': 'Vietnamese', 'task': 'transcribe'},
        )['text'].strip()
        reference = row.get('text', '') or ''
        asr_pairs.append({
            'index': index, 'reference_transcript': reference, 'asr_transcript': transcript,
            'wer': float(wer(reference, transcript)) if reference and transcript else None,
        })
        if (index + 1) % 25 == 0:
            print(f'ASR completed: {index + 1}/{ASR_SAMPLES}')
    del asr_pipe, asr_model
    free_gpu()
    asr_source = 'recomputed: leduckhai/MultiMed-ST whisper-small-vietnamese checkpoint-5000'
print('ASR transcripts:', len(asr_pairs), '|', asr_source)


def extract_entities(ner_pipe, text):
    entities = []
    for item in ner_pipe(text):
        label = item.get('entity_group') or item.get('entity')
        if label in {'0', 'O', 'dum'}:
            continue
        entities.append({'text': item['word'], 'label': label})
    return entities


def entity_counter(entities):
    return Counter((item['text'].strip().lower(), item['label']) for item in entities)


ner_sources = {'baseline': (XLMR_REPO, XLMR_SUBFOLDER), MODEL_KEY: (best_seed['checkpoint'], None)}
if phobert_available:
    ner_sources['phobert'] = (PHOBERT_CHECKPOINT, None)

asr_records = [dict(pair) for pair in asr_pairs]
for prefix, (path_or_repo, subfolder) in ner_sources.items():
    model, model_tokenizer = load_ner(path_or_repo, subfolder)
    ner_pipe = pipeline('ner', model=model, tokenizer=model_tokenizer, aggregation_strategy='simple', device=0)
    for record in asr_records:
        reference_entities = extract_entities(ner_pipe, record['reference_transcript'])
        asr_entities = extract_entities(ner_pipe, record['asr_transcript'])
        reference_keys, asr_keys = entity_counter(reference_entities), entity_counter(asr_entities)
        record[f'{prefix}_reference_entities'] = reference_entities
        record[f'{prefix}_asr_entities'] = asr_entities
        record[f'{prefix}_matched_entities'] = sum((reference_keys & asr_keys).values())
        record[f'{prefix}_reference_entity_count'] = sum(reference_keys.values())
    print('NER on ASR done:', prefix)
    del ner_pipe, model, model_tokenizer
    free_gpu()

(RESULT_DIR / 'asr_ner_comparison.jsonl').write_text(
    ''.join(json.dumps(to_jsonable(row), ensure_ascii=False) + '\n' for row in asr_records), encoding='utf-8',
)


def asr_summary(prefix):
    matched = sum(row[f'{prefix}_matched_entities'] for row in asr_records)
    reference_count = sum(row[f'{prefix}_reference_entity_count'] for row in asr_records)
    return {
        'matched_entities': matched,
        'reference_entity_count': reference_count,
        'entity_retention': matched / reference_count if reference_count else None,
    }


asr_summary_payload = {
    'num_samples': len(asr_records),
    'asr_source': asr_source,
    'baseline_xlm_roberta': asr_summary('baseline'),
    'vihealthbert_best_validation': asr_summary(MODEL_KEY),
    'note': 'All NER models process the identical ASR transcript. Retention compares each model prediction '
            'on reference vs ASR text and is not gold recall.',
}
if phobert_available:
    asr_summary_payload['phobert_best_validation'] = asr_summary('phobert')
elif (PHOBERT_RESULT_DIR / 'asr_ner_comparison_summary.json').exists() and not RUN_ASR and not SMOKE_TEST:
    saved_asr = json.loads((PHOBERT_RESULT_DIR / 'asr_ner_comparison_summary.json').read_text(encoding='utf-8'))
    asr_summary_payload['phobert_best_validation'] = saved_asr['phobert_best_validation']
    asr_summary_payload['phobert_source'] = 'saved summary (same cached transcripts)'
    print('Kiểm tra tái lập XLM-R: saved', round(saved_asr['baseline_xlm_roberta']['entity_retention'], 4),
          '| now', round(asr_summary_payload['baseline_xlm_roberta']['entity_retention'], 4))
(RESULT_DIR / 'asr_ner_comparison_summary.json').write_text(
    json.dumps(to_jsonable(asr_summary_payload), ensure_ascii=False, indent=2), encoding='utf-8',
)
print(json.dumps(to_jsonable(asr_summary_payload), ensure_ascii=False, indent=2))

## 6. Benchmark hiệu năng suy luận trên GPU

Đo trên cùng GPU, fp32 (giống dtype của demo): số tham số, dung lượng weight, độ trễ khi xử lý 1 câu (batch=1, p50/p95), thông lượng ở batch 32, và VRAM đỉnh. Dùng 300 câu test đầu tiên.

In [ ]:
BENCH_SENTENCES = 50 if SMOKE_TEST else 300
bench_words = [example['words'] for example in test_examples.select(range(min(BENCH_SENTENCES, len(test_examples))))]


def benchmark_model(path_or_repo, subfolder=None, batch_size=32, warmup=10):
    model, model_tokenizer = load_ner(path_or_repo, subfolder)
    model.eval()
    encoded = [encode_words(words, model_tokenizer)[0] for words in bench_words]
    pad_id = model_tokenizer.pad_token_id
    torch.cuda.reset_peak_memory_stats()

    def run(batch):
        width = max(len(item['input_ids']) for item in batch)
        ids = torch.full((len(batch), width), pad_id, dtype=torch.long)
        mask = torch.zeros((len(batch), width), dtype=torch.long)
        for row, item in enumerate(batch):
            ids[row, :len(item['input_ids'])] = torch.tensor(item['input_ids'])
            mask[row, :len(item['input_ids'])] = 1
        with torch.inference_mode():
            model(input_ids=ids.to(DEVICE), attention_mask=mask.to(DEVICE)).logits.argmax(-1).cpu()

    for item in encoded[:warmup]:
        run([item])
    torch.cuda.synchronize()
    latencies = []
    for item in encoded:
        start = time.perf_counter()
        run([item])
        torch.cuda.synchronize()
        latencies.append((time.perf_counter() - start) * 1000)
    start = time.perf_counter()
    for offset in range(0, len(encoded), batch_size):
        run(encoded[offset:offset + batch_size])
    torch.cuda.synchronize()
    throughput = len(encoded) / (time.perf_counter() - start)
    result = {
        'params_millions': sum(p.numel() for p in model.parameters()) / 1e6,
        'weights_mb_fp32': sum(p.numel() * p.element_size() for p in model.parameters()) / 1024**2,
        'avg_tokens_per_sentence': mean(len(item['input_ids']) for item in encoded),
        'latency_ms_p50': float(np.percentile(latencies, 50)),
        'latency_ms_p95': float(np.percentile(latencies, 95)),
        f'throughput_sentences_per_s_bs{batch_size}': throughput,
        'peak_vram_mb': torch.cuda.max_memory_allocated() / 1024**2,
        'gpu': gpu_name,
        'num_sentences': len(encoded),
    }
    del model
    free_gpu()
    return result


speed_benchmark = {'xlm_roberta_baseline': benchmark_model(XLMR_REPO, XLMR_SUBFOLDER)}
if phobert_available:
    speed_benchmark['phobert'] = benchmark_model(PHOBERT_CHECKPOINT)
speed_benchmark['vihealthbert'] = benchmark_model(best_seed['checkpoint'])
(RESULT_DIR / 'speed_benchmark.json').write_text(
    json.dumps(to_jsonable(speed_benchmark), ensure_ascii=False, indent=2), encoding='utf-8'
)
import pandas as pd
display(pd.DataFrame(speed_benchmark).T.round(2))

## 7. Bảng tổng hợp cho báo cáo

Gom số liệu của 3 model vào một bảng tổng và một bảng F1 theo từng loại entity. Kết quả được lưu ở `ketqua/comparison_table.md`. Số liệu PhoBERT lấy từ `giai_doan_13_ner_finetune_phobert/ketqua` nếu không có checkpoint để chấm lại.

In [ ]:
import pandas as pd

phobert_seed_summary = json.loads((PHOBERT_RESULT_DIR / 'best_config_seeds.json').read_text(encoding='utf-8'))


def pct(value):
    return None if value is None else round(100 * value, 2)


def entity_only_metrics(gold):
    '''Micro/macro chỉ trên entity thật (bỏ loại '_' sinh ra từ nhãn '0').
    Model chấm trên máy này có số chính xác; số PhoBERT đã lưu được suy ra từ classification report.'''
    if 'entity_only' in gold:
        return gold['entity_only']
    report = gold['classification_report']
    tp = pred = support = 0.0
    macro = []
    for name, row in report.items():
        if name.endswith(' avg') or name == '_':
            continue
        row_tp = row['recall'] * row['support']
        tp += row_tp
        pred += row_tp / row['precision'] if row['precision'] > 0 else 0.0
        support += row['support']
        macro.append(row['f1-score'])
    precision, recall = tp / pred if pred else 0.0, tp / support if support else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {'precision': precision, 'recall': recall, 'f1': f1, 'macro_f1': mean(macro), 'derived_from_report': True}


def summary_row(name, pretrain, gold, val_mean=None, val_std=None, retention=None, config=None):
    return {
        'Model': name, 'Pretrain': pretrain, 'Cấu hình chọn': config or '—',
        'Val F1 (3 seed)': f'{pct(val_mean)} ± {pct(val_std)}' if val_mean is not None else '— (dùng sẵn)',
        'Test P': pct(gold['precision']), 'Test R': pct(gold['recall']), 'Test F1 (micro)': pct(gold['f1']),
        'Test F1 (macro)': pct(gold['classification_report']['macro avg']['f1-score']),
        'Entity-only F1 (micro)': (f"≈{pct(entity_only_metrics(gold)['f1'])}" if entity_only_metrics(gold).get('derived_from_report')
                                   else pct(entity_only_metrics(gold)['f1'])),
        'Entity-only F1 (macro)': pct(entity_only_metrics(gold)['macro_f1']),
        'Retention sau ASR': pct(retention['entity_retention']) if retention else None,
    }


def config_text(config):
    return f"lr={config['learning_rate']:g}, ep={config['epochs']}, wd={config['weight_decay']}, seed={config['seed']}"


rows = [
    summary_row('XLM-RoBERTa-base (VietMed-NER, dùng sẵn)', 'Đa ngôn ngữ (~100 ngôn ngữ)',
                gold_metrics['baseline_xlm_roberta'],
                retention=asr_summary_payload.get('baseline_xlm_roberta')),
    summary_row('PhoBERT-base-v2 (nhóm fine-tune)', 'Tiếng Việt, văn bản chung',
                gold_metrics['phobert_best_validation_seed'],
                phobert_seed_summary['validation_f1_mean'], phobert_seed_summary['validation_f1_std_population'],
                asr_summary_payload.get('phobert_best_validation'),
                config_text({**phobert_seed_summary['best_config'], 'seed': phobert_seed_summary['selected_seed']['seed']})),
    summary_row('ViHealthBERT-base-syllable (nhóm fine-tune)', 'Tiếng Việt, văn bản y tế',
                gold_metrics['vihealthbert_best_validation_seed'],
                seed_summary['validation_f1_mean'], seed_summary['validation_f1_std_population'],
                asr_summary_payload.get('vihealthbert_best_validation'),
                config_text({**best_config, 'seed': best_seed['seed']})),
]
summary_df = pd.DataFrame(rows).set_index('Model')
display(summary_df)

entity_types = sorted(
    key for key in gold_metrics['vihealthbert_best_validation_seed']['classification_report']
    if not key.endswith(' avg') and key != '_'
)
per_entity_df = pd.DataFrame({
    'Support': [gold_metrics['vihealthbert_best_validation_seed']['classification_report'][t]['support'] for t in entity_types],
    'XLM-R F1': [pct(gold_metrics['baseline_xlm_roberta']['classification_report'].get(t, {}).get('f1-score', 0)) for t in entity_types],
    'PhoBERT F1': [pct(gold_metrics['phobert_best_validation_seed']['classification_report'].get(t, {}).get('f1-score', 0)) for t in entity_types],
    'ViHealthBERT F1': [pct(gold_metrics['vihealthbert_best_validation_seed']['classification_report'][t]['f1-score']) for t in entity_types],
}, index=entity_types)
per_entity_df['Δ ViHealthBERT − PhoBERT'] = (per_entity_df['ViHealthBERT F1'] - per_entity_df['PhoBERT F1']).round(2)
display(per_entity_df.sort_values('Support', ascending=False))

top_trials_df = pd.DataFrame([
    {'trial': t['trial_id'], 'val F1': pct(t['validation']['eval_f1']), 'giây': t['seconds'], 'VRAM GB': t['peak_vram_gb']}
    for t in trial_results[:5]
])
display(top_trials_df)

delta_f1 = gold_metrics['vihealthbert_best_validation_seed']['f1'] - gold_metrics['phobert_best_validation_seed']['f1']
markdown = '\n\n'.join([
    f'# So sánh 3 model NER trên VietMed-NER/test (n={len(test_examples)})',
    summary_df.to_markdown(),
    f"ViHealthBERT test F1 trên 3 seed: {pct(gold_payload['vihealthbert_all_seeds_test']['f1_mean'])} ± "
    f"{pct(gold_payload['vihealthbert_all_seeds_test']['f1_std_population'])} (chỉ để đo độ dao động; seed chính thức chọn theo validation).",
    f'Chênh lệch F1 micro ViHealthBERT − PhoBERT: {100 * delta_f1:+.2f} điểm phần trăm.',
    '## F1 theo loại entity', per_entity_df.sort_values('Support', ascending=False).to_markdown(),
    '## Hiệu năng suy luận (GPU)', pd.DataFrame(speed_benchmark).T.round(2).to_markdown(),
    '## Top 5 cấu hình validation của ViHealthBERT', top_trials_df.to_markdown(index=False),
    "Ghi chú: Test P/R/F1 giữ đúng cách chấm của notebook PhoBERT (nhãn '0' bị seqeval tính như entity loại '_'). "
    "Cột Entity-only đổi '0' → 'O' nên chỉ tính entity y tế thật; số có dấu ≈ được suy ra từ classification report đã lưu (micro lệch tối đa khoảng 1 điểm; macro chính xác). Đặt PHOBERT_CHECKPOINT để có số chính xác.",
    'Retention là độ nhất quán giữa dự đoán trên transcript ASR và transcript chuẩn, không phải recall gold.',
])
(RESULT_DIR / 'comparison_table.md').write_text(markdown + '\n', encoding='utf-8')
print('Đã lưu', RESULT_DIR / 'comparison_table.md')